# Lab 2 抓真資料、算均線、畫 K 線

Lab 1 用手打的 5 個收盤價算出 10/30 的 5 日均線 **2399.0**。
這個 Lab 從網路抓台積電 5 年的每日股價，讓電腦一次算出每一天的均線，再畫成 K 線圖。

> 資料：Yahoo 財經，用 `yfinance` 套件抓取。資料抓到你執行的那一天，所以筆數、最後一天的價格每次都可能不一樣；10/30 以前的數字不會變。

## 第 0 步：裝好指定版本的套件

套件＝別人寫好的工具包。Colab 已經內建，但版本會自己更新；先釘成跟這份講義一樣的版本，跑出來的結果才會跟講義一致。

> 預期：幾秒後格子左邊出現綠色勾勾，下面沒有任何訊息

In [ ]:
%pip install -q yfinance==0.2.66 plotly==5.24.1 pandas==2.2.3
# 💡 %pip install＝安裝套件；== 後面是版本；-q＝安靜模式，少印一點訊息

### 確認版本

> 預期：`0.2.66`，下一行 `5.24.1`

In [ ]:
import yfinance as yf     # 💡 import＝把套件拿進來用；as yf＝取個短名字，後面打 yf 就好
import plotly
print(yf.__version__)     # 💡 __version__＝套件的版本號
print(plotly.__version__)

## B1・抓資料：台積電 5 年的每日股價

### B1-1・抓下來，先看原始長相

> 預期：最下面 5 天的表格；表頭有**兩層**：上層是 `Close`、`High` 這些欄名，下層每欄都是 `2330.TW`

In [ ]:
stock_id = "2330"
df = yf.______(stock_id + ".TW", period="5y", auto_adjust=False)   # TODO：yf 裡「把股價抓下來」的工具
# 💡 stock_id + ".TW"＝Lab 1 的文字相加；period="5y"＝抓到今天為止的 5 年
# 💡 auto_adjust=False＝要「當天真的成交價」，跟看盤軟體一樣（不加的話，配息前的價格會被調低）
df.tail()       # 💡 df＝DataFrame（資料表）；tail()＝看最後 5 列

### B1-2・資料表的欄名

資料表由三個部分組成：**欄名**（最上面那排）、**列名**（最左邊那排）、中間的**值**。

> 預期：`MultiIndex`，每個欄名都是兩層，例如 `'Close', '2330.TW'`

In [ ]:
df._______     # TODO：資料表「欄名」的英文

### B1-3・資料表的列名

> 預期：`DatetimeIndex`，列名是一個一個日期

In [ ]:
df.index       # 💡 index＝列名；這份資料的列名是日期，後面畫 K 線的橫軸就用它

### B1-4・把兩層表頭攤平成一層

> 預期：表頭只剩一層：`Adj Close`、`Close`、`High`、`Low`、`Open`、`Volume`

In [ ]:
df.columns = df.columns.________(1)    # TODO：丟掉其中一層表頭的工具；1＝下層（位置從 0 數）
df.tail()                              # 💡 這樣之後打 df["Close"] 就拿得到收盤價

### B1-5・資料表有多大

> 預期：(1200 多, 6)：前面是幾列（幾個交易日），後面是幾欄；天數跟你執行的日期有關

In [ ]:
df._______     # TODO：一次看出資料表「幾列、幾欄」的屬性

## B2・清資料：刪掉不存在的交易日

### B2-1・每一天的成交量是不是 0？

10/23 證交所休市，沒有交易；但 Yahoo 的資料裡有這一天。

> 預期：一整欄的 True 或 False。畫面只顯示頭尾幾天，都是 `False`；10/23 那天是 True，藏在中間，下一格把它挑出來

In [ ]:
df["Volume"] _______ 0      # TODO：比較「等不等於」的符號（坑：一個 = 是存值，不是比較）

### B2-2・挑出成交量是 0 的日子

> 預期：只有 `2026-10-23` 這一列，四個價格都是 2385、`Volume` 是 0

In [ ]:
df[df["Volume"] _______ 0]    # TODO：跟上一格同一個比較；放進中括號就只留下 True 的列

### B2-3・刪掉它

> 這一格跑完不會有輸出，下一格再看結果。

In [ ]:
df = df[df[______] ____ 0]   # TODO：看哪一欄？要留下「大於 0」的列，用哪個比較符號？

### B2-4・再看一次大小

> 預期：比 B1-5 少 1 列，欄數一樣是 6

In [ ]:
df.shape       # 💡 不刪的話，均線會把休市那天也算成一天

## B3・均線：讓電腦算每一天

### B3-1・10/30 以前的最後 5 天

> 預期：5 個收盤價 `2385.0`、`2400.0`、`2425.0`、`2415.0`、`2370.0`，跟 Lab 1 手打的一樣

In [ ]:
df.loc[:"2026-10-30", "Close"].tail()
# 💡 loc[列, 欄]：列寫 :"2026-10-30"＝從頭到 10/30；欄寫 "Close"＝只要收盤價

### B3-2・5 日均線，對 Lab 1 的答案

> 預期：`2399.0`，跟 Lab 1 用 for 手算的一樣

In [ ]:
df["MA5"] = df["Close"].rolling(5).mean()   # 💡 rolling(5)＝每天往回框 5 天；mean()＝平均。新的一欄叫 MA5
print(df.loc["2026-10-30", "MA5"])          # 💡 拿 10/30 那一列的 MA5

### B3-3・20 日均線

天花板和地板要用 20 日均線算，先把它加成一欄。

> 預期：最後 5 天的 `Close`、`MA5`、`MA20` 三欄

In [ ]:
df["MA20"] = df["Close"].rolling(20).mean()   # 💡 跟上一格一樣，只是改框 20 天
df[["Close", "MA5", "MA20"]].tail()           # 💡 兩層中括號＝一次挑好幾欄

## C・畫 K 線圖

### C-1・最近半年的 K 線

> 預期：一張 K 線圖，**紅色是漲、綠色是跌**；滑鼠移到 K 棒上會顯示開高低收

In [ ]:
import plotly.graph_objects as go

recent = df.tail(120)          # 💡 最近 120 個交易日，大約半年
fig = go.Figure(go.Candlestick(
    x=recent.index,            # 💡 index＝每一列的日期
    open=recent["Open"],
    high=recent["High"],
    low=recent["Low"],
    close=recent["Close"],
    increasing_line_color="red",     # 💡 plotly 預設綠漲紅跌（歐美習慣），台灣是紅漲綠跌
    decreasing_line_color="green",
    name="日K",
))
fig.update_layout(title="台積電（2330）日K", xaxis_rangeslider_visible=False)  # 💡 關掉圖下方的縮放滑桿
fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])])                    # 💡 拿掉週末的空白
fig.show()

### C-2・疊上 20 日均線

> 預期：同一張 K 線圖上多一條 20 日均線

In [ ]:
fig.add_trace(go.Scatter(x=recent.index, y=recent["MA20"], name="20日均線",
                         line=dict(color="orange")))   # 💡 Scatter＝畫線；add_trace＝疊到同一張圖上
fig.show()

### ✏️ C 小作業：再疊上 5、60、120、240 日均線

20 日均線已經畫上去了，再把 5、60、120、240 日均線也加進 `df`，疊到同一張圖上。用 Lab 1 的 `for`，就不用同樣的程式寫 4 次。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
for n in [5, 60, 120, 240]:
    name = "MA" + str(n)
    df[name] = df["Close"].rolling(n).mean()

recent = df.tail(120)
colors = {5: "purple", 60: "blue", 120: "brown", 240: "black"}
for n in [5, 60, 120, 240]:
    name = "MA" + str(n)
    fig.add_trace(go.Scatter(x=recent.index, y=recent[name], name=str(n) + "日均線",
                             line=dict(color=colors[n])))
fig.show()
```

`recent` 要重新從 `df` 取一次，才會有新加的欄。
</details>

In [ ]:
# ✏️ C 小作業：你的答案
for n in [5, 60, 120, 240]:
    name = "MA" + str(n)                       # 💡 str(n)＝把數字變成文字，才能跟 "MA" 接起來
    df[name] = ____________                    # TODO：跟 B3 一樣算均線，只是天數換成 n

recent = df.tail(120)                          # 💡 recent 是加新欄之前取的，要重新取一次
colors = {5: "purple", 60: "blue", 120: "brown", 240: "black"}   # 💡 每條線指定顏色，避開紅綠（K 棒）和橘色（20 日）
for n in [5, 60, 120, 240]:
    name = "MA" + str(n)
    ____________                               # TODO：跟 C-2 一樣疊一條線；y 換成 recent[name]，顏色用 colors[n]
fig.show()

## 今天算出來的數字

| 學到的 | 做了什麼 |
|---|---|
| 抓資料 | 抓台積電 5 年日資料 |
| 欄名、列名、大小 | 看懂資料表長什麼樣子 |
| 用條件挑列 | 刪掉 10/23 休市那一列 |
| 移動平均 | 10/30 的 5 日均線 2399.0，跟 Lab 1 手算的一樣 |
| K 線圖 | 紅漲綠跌的 K 線圖＋均線 |

下一個 Lab：用 Lab 1 的 `bias` 算出每一天的乖離率，找出台積電的天花板和地板。